<a href="https://colab.research.google.com/github/evgeniy-borisov/vairl/blob/main/notebooks/hypothesis-synthesis-agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Синтез гипотез для повышения качества агентов

Локальная LLM → структурированные гипотезы → ChromaDB → оценка → визуализация пространства.

**VAIRL** — Virtual AI Research Lab

> Перед запуском: **Runtime → Change runtime type → T4 GPU** (рекомендуется).

## 1. Установка зависимостей

In [ ]:
!pip install -q chromadb sentence-transformers matplotlib scikit-learn

import torch
HAS_GPU = torch.cuda.is_available()
print(f"GPU: {HAS_GPU}", torch.cuda.get_device_name(0) if HAS_GPU else "CPU only")

if HAS_GPU:
    !pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

## 2. Конфигурация задачи

Гипотезы описывают **изменения в архитектуре или политике агента**, которые можно проверить экспериментом.

In [ ]:
import json
import re
import uuid
from dataclasses import dataclass, asdict
from typing import Any

TASK = (
    "Повысить качество LLM-агента: надёжность вызова инструментов, "
    "глубину планирования, восстановление после ошибок и согласованность многошаговых задач."
)

AGENT_QUALITY_AXES = [
    "tool_use",
    "planning",
    "memory",
    "reflection",
    "error_recovery",
    "latency_tradeoff",
]

N_HYPOTHESES_PER_ROUND = 6
N_ROUNDS = 2
USE_LOCAL_LLM = HAS_GPU  # без GPU — демо-режим с шаблонными гипотезами

HYPOTHESIS_EXAMPLE = {
    "claim": "Явный шаг рефлексии перед вызовом инструмента снижает долю неверных аргументов",
    "mechanism": "Агент проверяет схему API и контекст до исполнения",
    "axis": "tool_use",
    "variables": {"reflection_before_tool": True, "max_retries": 2},
    "metric": "tool_call_success_rate",
    "prediction": "рост success rate на 5–15%",
    "falsification": "нет статистически значимого улучшения на 50 эпизодах",
}

print("Задача:", TASK)
print("Оси качества:", ", ".join(AGENT_QUALITY_AXES))

## 3. Генерация гипотез

С GPU — локальная **Qwen2.5-7B-Instruct** (GGUF Q4). Без GPU — демо-шаблоны, чтобы пройти весь пайплайн.

In [ ]:
def extract_json_array(text: str) -> list[dict[str, Any]]:
    match = re.search(r"\[.*\]", text, flags=re.DOTALL)
    if not match:
        raise ValueError("JSON-массив не найден в ответе LLM")
    return json.loads(match.group())


def normalize_hypothesis(raw: dict[str, Any], idx: int) -> dict[str, Any]:
    axis = raw.get("axis", AGENT_QUALITY_AXES[idx % len(AGENT_QUALITY_AXES)])
    if axis not in AGENT_QUALITY_AXES:
        axis = AGENT_QUALITY_AXES[0]
    return {
        "id": raw.get("id") or f"h-{uuid.uuid4().hex[:8]}",
        "claim": str(raw.get("claim", "")).strip(),
        "mechanism": str(raw.get("mechanism", "")).strip(),
        "axis": axis,
        "variables": raw.get("variables") or {},
        "metric": str(raw.get("metric", "task_success_rate")).strip(),
        "prediction": str(raw.get("prediction", "")).strip(),
        "falsification": str(raw.get("falsification", "")).strip(),
    }


def demo_hypotheses(n: int, round_idx: int) -> list[dict[str, Any]]:
    templates = [
        {
            "claim": "Декомпозиция цели на подзадачи перед вызовом инструментов повышает success rate",
            "mechanism": "Явный план снижает хаотичные tool calls",
            "axis": "planning",
            "variables": {"planner": "hierarchical", "max_subtasks": 5},
            "metric": "task_success_rate",
            "prediction": "+8–12% на многошаговых бенчмарках",
            "falsification": "нет улучшения на SWE-bench lite",
        },
        {
            "claim": "Сжатое episodic memory снижает потерю контекста в длинных сессиях",
            "mechanism": "Периодическая суммаризация сохраняет ключевые факты",
            "axis": "memory",
            "variables": {"summarize_every_n_turns": 6},
            "metric": "context_retention_score",
            "prediction": "меньше повторных запросов к пользователю",
            "falsification": "retention не растёт на диалогах >30 turns",
        },
        {
            "claim": "Retry с анализом traceback улучшает восстановление после ошибок API",
            "mechanism": "Агент классифицирует тип ошибки и меняет стратегию",
            "axis": "error_recovery",
            "variables": {"error_classifier": True, "max_retries": 3},
            "metric": "recovery_rate",
            "prediction": "+10% успешных восстановлений",
            "falsification": "recovery rate не меняется на synthetic fault injection",
        },
        {
            "claim": "Критик-верификатор второго прохода отсекает галлюцинированные факты",
            "mechanism": "Отдельный проход сверяет ответ с retrieved context",
            "axis": "reflection",
            "variables": {"critic_pass": True},
            "metric": "hallucination_rate",
            "prediction": "снижение hallucination rate на 20%",
            "falsification": "критик не снижает false claims в RAG QA",
        },
        {
            "claim": "Кэширование результатов инструментов снижает latency без потери качества",
            "mechanism": "Идемпотентные вызовы переиспользуются в рамках эпизода",
            "axis": "latency_tradeoff",
            "variables": {"tool_cache_ttl_s": 120},
            "metric": "p95_latency",
            "prediction": "−30% p95 при том же success rate",
            "falsification": "success rate падает из-за устаревшего кэша",
        },
        {
            "claim": "Структурированный JSON-вывод для tool args повышает валидность вызовов",
            "mechanism": "Grammar-constrained decoding уменьшает синтаксические ошибки",
            "axis": "tool_use",
            "variables": {"json_schema_enforced": True},
            "metric": "tool_call_success_rate",
            "prediction": "+15% валидных вызовов",
            "falsification": "нет эффекта на function-calling benchmark",
        },
    ]
    out = []
    for i in range(n):
        t = templates[(i + round_idx) % len(templates)].copy()
        t["id"] = f"demo-r{round_idx}-{i}"
        out.append(normalize_hypothesis(t, i))
    return out


llm = None
if USE_LOCAL_LLM:
    from llama_cpp import Llama

    llm = Llama.from_pretrained(
        repo_id="Qwen/Qwen2.5-7B-Instruct-GGUF",
        filename="*Q4_K_M*.gguf",
        n_gpu_layers=-1,
        n_ctx=4096,
        verbose=False,
    )


def build_prompt(task: str, n: int, prior: list[dict[str, Any]] | None = None) -> str:
    prior_txt = ""
    if prior:
        prior_txt = "\nУже проверенные гипотезы (не дублируй):\n" + json.dumps(
            [{"claim": h["claim"], "axis": h["axis"]} for h in prior],
            ensure_ascii=False,
            indent=2,
        )
    return f"""Сгенерируй {n} проверяемых гипотез для улучшения LLM-агента.

Задача: {task}

Оси качества (поле axis): {', '.join(AGENT_QUALITY_AXES)}

Пример одной гипотезы:
{json.dumps(HYPOTHESIS_EXAMPLE, ensure_ascii=False, indent=2)}
{prior_txt}

Ответ — ТОЛЬКО JSON-массив объектов с полями:
id, claim, mechanism, axis, variables, metric, prediction, falsification
"""


def generate_hypotheses(
    n: int,
    round_idx: int = 0,
    prior: list[dict[str, Any]] | None = None,
) -> list[dict[str, Any]]:
    if not USE_LOCAL_LLM:
        return demo_hypotheses(n, round_idx)

    prompt = build_prompt(TASK, n, prior)
    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=2048,
    )
    text = response["choices"][0]["message"]["content"]
    raw_list = extract_json_array(text)
    return [normalize_hypothesis(item, i) for i, item in enumerate(raw_list[:n])]


seed = generate_hypotheses(N_HYPOTHESES_PER_ROUND, round_idx=0)
print(f"Сгенерировано: {len(seed)} гипотез (режим: {'LLM' if USE_LOCAL_LLM else 'demo'})")
print(json.dumps(seed[0], ensure_ascii=False, indent=2))

## 4. ChromaDB: хранение и дедупликация

Эмбеддинги помогают не генерировать семантически близкие гипотезы повторно.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="intfloat/multilingual-e5-small"
)

client = chromadb.Client()
collection = client.get_or_create_collection(
    name="agent_hypotheses",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)

SIMILARITY_THRESHOLD = 0.88  # 1 - cosine distance


def hypothesis_document(h: dict[str, Any]) -> str:
    return (
        f"passage: {h['claim']} | {h['mechanism']} | axis={h['axis']} | "
        f"metric={h['metric']} | prediction={h['prediction']}"
    )


def is_duplicate(h: dict[str, Any]) -> bool:
    if collection.count() == 0:
        return False
    result = collection.query(
        query_texts=[hypothesis_document(h)],
        n_results=1,
    )
    distance = result["distances"][0][0]
    similarity = 1.0 - distance
    return similarity >= SIMILARITY_THRESHOLD


def store_hypothesis(h: dict[str, Any]) -> bool:
    if is_duplicate(h):
        return False
    collection.add(
        ids=[h["id"]],
        documents=[hypothesis_document(h)],
        metadatas=[{
            "axis": h["axis"],
            "metric": h["metric"],
            "claim": h["claim"][:500],
        }],
    )
    return True


all_hypotheses: list[dict[str, Any]] = []
for h in seed:
    if store_hypothesis(h):
        all_hypotheses.append(h)

print(f"В коллекции: {collection.count()} уникальных гипотез")

## 5. Быстрая эвристическая оценка

Замените `score_hypothesis` на реальный бенчмарк агента (SWE-bench, ToolBench, custom env).

In [ ]:
import random

AXIS_WEIGHTS = {
    "tool_use": 1.0,
    "planning": 0.95,
    "error_recovery": 0.9,
    "reflection": 0.85,
    "memory": 0.8,
    "latency_tradeoff": 0.7,
}


def score_hypothesis(h: dict[str, Any]) -> float:
    """Эвристика для демо. В проде — запуск агента с variables из гипотезы."""
    base = AXIS_WEIGHTS.get(h["axis"], 0.5)
    specificity = min(len(h["claim"].split()), 20) / 20.0
    falsifiable = 0.15 if h["falsification"] else 0.0
    measurable = 0.15 if h["metric"] else 0.0
    noise = random.uniform(-0.05, 0.05)
    return round(base * 0.5 + specificity * 0.3 + falsifiable + measurable + noise, 3)


for h in all_hypotheses:
    h["score"] = score_hypothesis(h)

ranked = sorted(all_hypotheses, key=lambda x: x["score"], reverse=True)
print("Топ-3 гипотезы:")
for h in ranked[:3]:
    print(f"  [{h['score']}] {h['axis']}: {h['claim'][:80]}...")

## 6. Итеративный цикл refine

Лучшие гипотезы прошлого раунда подаются в контекст следующей генерации.

In [ ]:
for round_idx in range(1, N_ROUNDS):
    prior = ranked[:3]
    batch = generate_hypotheses(N_HYPOTHESES_PER_ROUND, round_idx=round_idx, prior=prior)
    added = 0
    for h in batch:
        h["score"] = score_hypothesis(h)
        if store_hypothesis(h):
            all_hypotheses.append(h)
            added += 1
    ranked = sorted(all_hypotheses, key=lambda x: x["score"], reverse=True)
    print(f"Раунд {round_idx}: +{added} новых, всего {len(all_hypotheses)}")

## 7. Визуализация пространства гипотез

PCA по эмбеддингам ChromaDB + цвет по оси качества, размер — по score.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

ids = [h["id"] for h in all_hypotheses]
stored = collection.get(ids=ids, include=["embeddings"])
embeddings = np.array(stored["embeddings"])

coords = PCA(n_components=2, random_state=42).fit_transform(embeddings)

axis_colors = {ax: i for i, ax in enumerate(AGENT_QUALITY_AXES)}
colors = [axis_colors.get(h["axis"], 0) for h in all_hypotheses]
sizes = [80 + 320 * h["score"] for h in all_hypotheses]

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=sizes, alpha=0.75, cmap="tab10")
for i, h in enumerate(all_hypotheses):
    if h["score"] >= sorted(x["score"] for x in all_hypotheses, reverse=True)[2]:
        ax.annotate(h["axis"], (coords[i, 0], coords[i, 1]), fontsize=8, alpha=0.8)
ax.set_title("Пространство гипотез (PCA эмбеддингов)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()

print("\n=== Итоговый рейтинг ===")
for i, h in enumerate(ranked, 1):
    print(f"{i}. [{h['score']}] {h['axis']}: {h['claim']}")

## 8. Следующие шаги

1. Подключить реальный бенчмарк агента вместо `score_hypothesis`
2. Сохранять коллекцию на Google Drive: `PersistentClient(path="/content/drive/MyDrive/chroma")`
3. Заменить PCA на [PaCMAP](https://github.com/YingfanWang/PaCMAP) для лучшей глобальной структуры
4. Добавить нейросимволического критика (проверка falsification criteria)

Статья на сайте VAIRL: [Синтез гипотез локальной LLM для повышения качества агентов](https://evgeniy-borisov.github.io/vairl/blog/2026/06/26/llm-hypothesis-synthesis-agents/)